In [1]:
# imports
import MDAnalysis as mda
import pandas as pd
import nglview as nv
from tqdm.notebook import tqdm
from setup_structures import assign_segment_ids, identify_subaggregates
from merge_structures import merge_structures

In [ ]:
proteins = ["hpl2", "lin13", "met2", "lin65"]
protein_file_names = {prot: f"inputs/single_comp_structures/{prot}.pdb" for prot in proteins}
protein_dimensions = {prot: [600,600,3000,90,90,90] for prot in proteins}

In [ ]:
# # other input
# proteins = ["hpl2_lin13", "met2_lin65"]
# protein_file_names = {prot: f"inputs/multiple_comp_structures/{prot}.pdb" for prot in proteins}
# protein_dimensions = {prot: [600,600,3000,90,90,90] for prot in proteins}

In [4]:
# load the structure files, assign unique segment IDs for every protein, identify aggregates in the structures
# and save relevant paramenters in a pandas dataframe
existing_segids = {}
protein_data = {}
for prot in tqdm(protein_file_names, desc="Loading structure files", leave=False, unit="Files"):
    x = mda.Universe(protein_file_names[prot])
    u = x.copy()
    tqdm.write(f"Created universe for {prot}")

    # Add unique segment IDs to differentiate between protein chains
    u, prefix = assign_segment_ids(u, prot, existing_segids)
    tqdm.write(f"Assigned segment IDs to {prot}")

    # assign known universe dimensions
    u.dimensions = protein_dimensions[prot]

    # Analyze chains and clusters
    clusters = identify_subaggregates(u)
    tqdm.write(f"Identified clusters in {prot}")
    protein_data[prot] = {
        'segment_prefix': prefix,
        'universe': u,
        "n_atoms": len(u.atoms),
        'subaggregates': clusters,
        'rg_min_cluster': min([cluster["cluster"].radius_of_gyration() for cluster in clusters]),
        'rg_max': max([prot.atoms.radius_of_gyration() for prot in u.segments]),
        "n_proteins": len(u.segments), 
        "box_dimensions": u.dimensions
    }

proteins_df = pd.DataFrame.from_dict(protein_data, orient='index')
proteins_df

Loading structure files:   0%|          | 0/4 [00:00<?, ?Files/s]

FileNotFoundError: [Errno 2] No such file or directory: 'input_structures/single_comp_structures/hpl2.pdb'

In [ ]:
view = nv.show_mdanalysis(proteins_df.loc["hpl2_lin13", "universe"])
                          
# add the unit cell (the box)
view.add_unitcell()

view

In [ ]:
merged = merge_structures(proteins_df)

In [ ]:
view = nv.show_mdanalysis(merged.atoms)

# add the unit cell (the box)
view.add_unitcell()

view

In [ ]:
from MDAnalysis.lib.distances import capped_distance

def check_true_structure_overlaps(u, cutoff=10.0):
    """
    Groups segments by their 2-character prefix and checks for 
    collisions between these groups (ignoring internal chain-chain contacts).
    """
    # 1. Group segments by their prefix (first 2 chars)
    # Result: {'H2': AtomGroup, 'L1': AtomGroup, ...}
    prefixes = sorted(list(set(s.segid[:2] for s in u.segments)))
    structure_groups = {p: u.select_atoms(f"segid {p}*") for p in prefixes}
    
    total_collisions = 0
    checked_pairs = []

    # 2. Nested loop to check only DIFFERENT structures
    for i, pref1 in enumerate(prefixes):
        for j, pref2 in enumerate(prefixes):
            if i >= j: continue # Avoid double-counting and self-interaction
            
            group1 = structure_groups[pref1]
            group2 = structure_groups[pref2]
            
            # Use capped_distance for the two distinct AtomGroups
            pairs = capped_distance(group1.positions, 
                                    group2.positions, 
                                    max_cutoff=cutoff, 
                                    box=u.dimensions)
            
            count = len(pairs[0])
            if count > 0:
                print(f"Collision: {pref1} <-> {pref2} | {count} atoms overlapping")
            
            total_collisions += count
            
    return total_collisions

# Run the check
true_overlaps = check_true_structure_overlaps(merged)
print(f"--- Final Report: {true_overlaps} true inter-structure collisions found ---")

In [ ]:
from pathlib import Path
required_order = ["A", "B", "C", "D"]
sorted_segments = []
for seg_prefix in required_order:
    for seg_idx in range(1, 41):
        seg_id = f"{seg_prefix}{seg_idx:03d}"
        # select protein by protein in order of the original structures
        selection = merged.select_atoms(f"segid {seg_id}")
        
        if len(selection) > 0:
            sorted_segments.append(selection)
            print(f"Atoms added for {seg_id}") 
        else:
            print(f"Warning: No atoms found for {seg_id}") 

ordered_merged = mda.Merge(*sorted_segments)
ordered_merged.dimensions = merged.dimensions
output_path = Path("outputs/merged_structure_ordered.pdb")
ordered_merged.atoms.write(output_path)

In [ ]:
from openmm import app, unit, XmlSerializer, Platform
import openmm as mm

# OPENMM ENERGY EVALUATION
def make_energy_sim(system_xml: Path, top_pdb: Path, T, platform_name="CPU", gamma=0.01, dt_ps=0.01):
    system = XmlSerializer.deserialize(system_xml.read_text())
    pdb = app.PDBFile(str(top_pdb))
    
    # Setup integrator
    integ = mm.LangevinMiddleIntegrator(T*unit.kelvin, gamma/unit.picosecond, dt_ps*unit.picoseconds)
    
    # Set platform (use 'Reference' if 'CPU' is not found, or 'CUDA' if available)
    try:
        platform = Platform.getPlatformByName(platform_name)
    except:
        platform = Platform.getPlatformByName("Reference")
        
    sim = app.Simulation(pdb.topology, system, integ, platform)
    
    # IMPORTANT: Manually set box vectors from MDAnalysis dimensions [L, L, L, 90, 90, 90]
    box_lx, box_ly, box_lz = ordered_merged.dimensions[:3]
    sim.context.setPeriodicBoxVectors(
        [box_lx/10, 0, 0], # Å to nm
        [0, box_ly/10, 0], 
        [0, 0, box_lz/10]
    )
    
    return sim

# ... (your merge code here) ...

# 1. Initialize the simulation as before
sim = make_energy_sim(
    system_xml=Path("inputs/all_comps_condensed_260.15K.xml"), 
    top_pdb=output_path, 
    T=260.15
)

# 2. INSTEAD OF RELOADING THE PDB: 
# Get positions directly from MDAnalysis (ordered_merged)
# We convert Ångström to Nanometers (required by OpenMM)
mda_positions = ordered_merged.atoms.positions / 10.0

# 3. Set positions directly into the context
# This ensures the counts are EXACTLY the same as what you merged
sim.context.setPositions(mda_positions)

# 4. Calculate Potential Energy
st = sim.context.getState(getEnergy=True)
pot_energy = st.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)

print(f"--- SUCCESS ---")
print(f"Atom count in MDA: {len(ordered_merged.atoms)}")
print(f"Potential Energy: {pot_energy:.2f} kJ/mol")


In [ ]:
print(f"MDA Atoms: {len(ordered_merged.atoms)}")
print(f"XML Atoms: {sim.system.getNumParticles()}")